In [45]:
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

import sys; sys.path.append("..") #to direct program to the root files
from utils.get_data import get_processed_files

from datetime import datetime

In [ ]:
"""
Only run if there changes to ../utils/get_data folder
import importlib
import utils.get_data
importlib.reload(utils.get_data) #update with any changes made
"""

'\nOnly run if there changes to utils.get_data folder\nimport importlib\nimport utils.get_data\nimportlib.reload(utils.get_data) #update with any changes made\n'

In [47]:
# below are estimated date
TERRA_LUNA_CRASH_START_DATE = datetime.strptime("7 May 2022", "%d %B %Y")
GFC_CRASH_START_DATE = datetime.strptime("15 September 2008", "%d %B %Y")

In [3]:
erc_20_dfs = get_processed_files("..\\datasets\\processed\\erc_20")
erc_20_dfs

{'dai_price_data':      Unnamed: 0   timestamp    open  high     low   close        date
 0             0  1648857600  0.9999   1.0  0.9989  1.0000  2022-04-02
 1             1  1648944000  1.0000   1.0  0.9989  0.9990  2022-04-03
 2             2  1649030400  0.9990   1.0  0.9989  0.9995  2022-04-04
 3             3  1649116800  0.9994   1.0  0.9987  0.9999  2022-04-05
 4             4  1649203200  0.9999   1.0  0.9988  0.9991  2022-04-06
 ..          ...         ...     ...   ...     ...     ...         ...
 210         210  1667001600  0.9997   1.0  0.9987  1.0000  2022-10-29
 211         211  1667088000  1.0000   1.0  0.9991  1.0000  2022-10-30
 212         212  1667174400  1.0000   1.0  0.9989  1.0000  2022-10-31
 213         213  1667260800  1.0000   1.0  0.9983  0.9999  2022-11-01
 214         214  1667347200  0.9999   1.0  0.9990  0.9999  2022-11-02
 
 [215 rows x 7 columns],
 'event_data':     Unnamed: 0                                              event   timestamp  \
 0     

In [4]:
gfc_dfs = get_processed_files("..\\datasets\\processed\\gfc_data")
gfc_dfs

{'AIG':            Price       Close        High         Low        Open    Volume  \
 0     2005-01-04  792.226868  799.999660  790.552699  793.422682    379110   
 1     2005-01-05  805.380432  812.674866  796.651008  797.607644    558800   
 2     2005-01-06  806.695984  812.435949  799.999456  801.195269    408765   
 3     2005-01-07  808.130920  811.718362  804.304302  807.772191    311070   
 4     2005-01-10  809.565979  811.838055  804.782724  807.413529    253345   
 ...          ...         ...         ...         ...         ...       ...   
 2006  2012-12-21   26.324486   26.665474   26.021382   26.460879  31856600   
 2007  2012-12-24   26.673058   26.673058   26.172938   26.241136   6816400   
 2008  2012-12-26   26.786713   26.847335   26.491188   26.756405  11728100   
 2009  2012-12-27   26.498762   26.900372   26.066841   26.900372  16674800   
 2010  2012-12-28   26.203245   26.513925   26.142623   26.218398  11919300   
 
      ticker        date  
 0       AIG  20

In [ ]:
def create_candleLit_graph(df, name):
    """
    Show the trend of high, open, low, close prices

    Args:
        df -> pd.DataFrame
            should consist of the following columns ("open", "high", "low", "close", "date")
        name -> String, Name of the coin , or stocks , etc
    Output:
        candlestick chart
    """
    try :
        fig = go.Figure(data=[go.Candlestick(
            x=df.date,
            open=df['Open'],
            high=df['High'],
            low=df['Low'],
            close=df['Close'],
            increasing_line_color='green',
            decreasing_line_color='red'
        )])

        fig.add_shape(
            type="line", x0=TERRA_LUNA_CRASH_START_DATE, x1=TERRA_LUNA_CRASH_START_DATE, y0=df.Low.min(), y1=df.High.max() + 0.2,
            line=dict(color="red", width=1, dash="dash")
        )

        fig.update_layout(title=f'Price changes for {name}', xaxis_title='Date', yaxis_title="Prices")
        fig.show()
    except :
        print("unable to create candlestick chart")
        print("columns in df", df.columns.tolist())
        return

def create_line_graph(df, x, y, hue):
    """
    Analysing how input x column differ across y and for each hue
    using Line graph

    Args:
        df:
            pd.DataFrame (consisting of x,y,hue columns)
        x:
            string (name of column to be displayed as x-axis on line graph)
        y:
            string (name of column to be displayed as y-axis on line graph)
        hue:
            string (name of column to be displayed as hue on line graph)

    Output:
        Line graph figure
    """
    fig = px.line(df, x=x, y=y, color=hue)
    fig.add_shape(
        type="line", x0=TERRA_LUNA_CRASH_START_DATE, x1=TERRA_LUNA_CRASH_START_DATE, y0=df[y].min(), y1=df[y].max(),
        line=dict(color="red", width=1, dash="dash")
    )
    fig.update_layout(title=f'{y} Price changes across time', xaxis_title=x, yaxis_title=f"{y} prices")
    fig.show()



### Concat related files
- concat all price_data
- concat all gfc data

In [35]:
price_data = None
for name, df in erc_20_dfs.items():
    if name.endswith("price_data"):
        print("merging ", name)
        df["coins"] = name.split("_")[0] #because file name is processed_<coin_name>_price_data.csv
        df = df.rename(columns={'open': 'Open', 'low': 'Low', 'close': 'Close', 'high': 'High'})
        if (price_data is None):
            price_data = df
        else:
            price_data = pd.concat([price_data, df])

        print("analysing individual candle chart")
        create_candleLit_graph(df, name.split("_")[0])
    if "Unnamed: 0" in df.columns.tolist(): #drop unnamed column
        df.drop("Unnamed: 0", axis=1, inplace=True)

merging  dai_price_data
analysing individual candle chart


merging  pax_price_data
analysing individual candle chart


merging  usdc_price_data
analysing individual candle chart


merging  usdt_price_data
analysing individual candle chart


merging  ustc_price_data
analysing individual candle chart


merging  wluna_price_data
analysing individual candle chart


In [7]:
gfc_data = None
for name, df in gfc_dfs.items():
    print("merging ", name)
    if (gfc_data is None):
        gfc_data = df
    else:
        gfc_data = pd.concat([gfc_data, df])

    print("analysing individual candle chart")
    create_candleLit_graph(df, name)
    if "Unnamed: 0" in df.columns.tolist(): #drop unnamed column
        df.drop("Unnamed: 0", axis=1, inplace=True)

merging  AIG
analysing individual candle chart


merging  C
analysing individual candle chart


merging  JPM
analysing individual candle chart


merging  TEDRATE
analysing individual candle chart
unable to create candlestick chart
columns in df ['observation_date', 'TEDRATE', 'ticker', 'date']
merging  WGS3MO
analysing individual candle chart
unable to create candlestick chart
columns in df ['observation_date', 'WGS3MO', 'ticker', 'date']
merging  ^DJI
analysing individual candle chart


merging  ^GSPC
analysing individual candle chart


merging  ^VIX
analysing individual candle chart


In [ ]:
gfc_data

In [ ]:
price_data

# Analysing data

In [14]:
gfc_data

,Price,Close,High,Low,Open,Volume,ticker,date,observation_date,TEDRATE,WGS3MO
0,2005-01-04,792.226868,799.999660,790.552699,793.422682,379110.0,AIG,2005-01-04,NaN,NaN,NaN
1,2005-01-05,805.380432,812.674866,796.651008,797.607644,558800.0,AIG,2005-01-05,NaN,NaN,NaN
2,2005-01-06,806.695984,812.435949,799.999456,801.195269,408765.0,AIG,2005-01-06,NaN,NaN,NaN
3,2005-01-07,808.130920,811.718362,804.304302,807.772191,311070.0,AIG,2005-01-07,NaN,NaN,NaN
4,2005-01-10,809.565979,811.838055,804.782724,807.413529,253345.0,AIG,2005-01-10,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2006,2012-12-21,17.840000,19.930000,17.760000,19.850000,0.0,^VIX,2012-12-21,NaN,NaN,NaN
2007,2012-12-24,17.840000,18.660000,17.840000,18.459999,0.0,^VIX,2012-12-24,NaN,NaN,NaN
2008,2012-12-26,19.480000,19.629999,18.610001,18.709999,0.0,^VIX,2012-12-26,NaN,NaN,NaN
2009,2012-12-27,19.469999,20.900000,19.110001,19.389999,0.0,^VIX,2012-12-27,NaN,NaN,NaN


In [17]:
price_data

,timestamp,Open,High,Low,Close,date,coins
0,1648857600,0.999900,1.000000,0.998900,1.000000,2022-04-02,dai
1,1648944000,1.000000,1.000000,0.998900,0.999000,2022-04-03,dai
2,1649030400,0.999000,1.000000,0.998900,0.999500,2022-04-04,dai
3,1649116800,0.999400,1.000000,0.998700,0.999900,2022-04-05,dai
4,1649203200,0.999900,1.000000,0.998800,0.999100,2022-04-06,dai
...,...,...,...,...,...,...,...
210,1667001600,0.000239,0.000252,0.000235,0.000241,2022-10-29,wluna
211,1667088000,0.000241,0.000249,0.000229,0.000236,2022-10-30,wluna
212,1667174400,0.000236,0.000268,0.000230,0.000244,2022-10-31,wluna
213,1667260800,0.000244,0.000255,0.000233,0.000236,2022-11-01,wluna


## Analysing coin prices for ERC20

In [58]:
create_line_graph(price_data[price_data['coins'] != "wluna"], 'date', 'Open', 'coins')
create_line_graph(price_data[price_data['coins'] != "wluna"], 'date', 'Close', 'coins')
create_line_graph(price_data[price_data['coins'] != "wluna"], 'date', 'High', 'coins')
create_line_graph(price_data[price_data['coins'] != "wluna"], 'date', 'Low', 'coins')


In [61]:
create_line_graph(price_data[~price_data['coins'].isin(["wluna", "ustc"])], 'date', 'Open', 'coins')
create_line_graph(price_data[~price_data['coins'].isin(["wluna", "ustc"])], 'date', 'Close', 'coins')
create_line_graph(price_data[~price_data['coins'].isin(["wluna", "ustc"])], 'date', 'High', 'coins')
create_line_graph(price_data[~price_data['coins'].isin(["wluna", "ustc"])], 'date', 'Low', 'coins')
